# 4. Exports

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khoshkhah/duckOSM/blob/main/docs/notebooks/04_exports.ipynb) <a href="https://khoshkhah.github.io/duckOSM/notebooks/04_exports.ipynb" download><img src="https://img.shields.io/badge/Download-notebook-F37626?logo=jupyter&logoColor=white" alt="Download the notebook"></a>

You export the network of Monaco to six formats. Each cell writes the files and does a check on them.

| Format | Method | Check |
|---|---|---|
| SUMO | `net.to_sumo` | SUMO's `netconvert` reads the plain XML and writes `.net.xml` |
| GMNS | `net.to_gmns` | `duckosm.check_gmns`: the rules of the GMNS tables |
| MATSim | `net.to_matsim` | the XML has one link for each edge |
| OpenDRIVE | `net.to_opendrive` | the XML has one road for each edge |
| GeoPackage | `net.to_gis` | DuckDB reads each layer |
| QuackOSM table, PMTiles | `duckosm.quack`, `duckosm quack --pmtiles` | DuckDB reads the table; the PMTiles header |

In [ ]:
# Install duckOSM when it is not installed (for example in Colab). It takes approximately 1 minute.
try:
    import duckosm
except ImportError:
    %pip install -q "duckosm[widget,sumo]" roadstyle
    import duckosm
print("duckosm", duckosm.__version__)

In [ ]:
area = duckosm.get("Monaco", out="data")      # the OSM file of Monaco and its boundary
net = duckosm.build(pbf=area.pbf, boundary=area.boundary, modes=["driving"],
                    output="data/monaco.duckdb")   # the driving network: a duckosm.Network
net

In [ ]:
from pathlib import Path


def files(folder):
    """Print the files of a folder and their sizes."""
    for f in sorted(Path(folder).glob("*")):
        print(f"{f.stat().st_size / 1e6:8.2f} MB  {f}")


n_edges = len(net.edges())    # the number of driving edges, to compare with each export
n_edges

## SUMO

`to_sumo` writes the plain XML files and runs SUMO's `netconvert` on them. The extra `sumo` installs `netconvert`.

In [ ]:
import os, shutil, sys

os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]   # netconvert of the extra sumo
sumo = net.to_sumo("out/sumo", run_netconvert=shutil.which("netconvert") is not None)
files("out/sumo")
print(sumo["n_edges"], "edges,", sumo["n_connections"], "connections, netconvert wrote:", sumo.get("net"))

## GMNS

In [ ]:
net.to_gmns("out/gmns.duckdb", to_csv="out/gmns")   # a GMNS database and its CSV files
files("out/gmns")
checks = duckosm.check_gmns("out/gmns.duckdb")    # (schema, check, bad rows, rows) for each check
failed = [c for c in checks if c[2] > 0]
print(len(checks), "checks,", len(failed), "with failures")

## MATSim

In [ ]:
import gzip
import xml.etree.ElementTree as ET

net.to_matsim("out/matsim_network.xml.gz")
links = ET.parse(gzip.open("out/matsim_network.xml.gz")).findall(".//link")    # the links of the network file
print(f"{Path('out/matsim_network.xml.gz').stat().st_size / 1e6:.2f} MB,", len(links), "links,", n_edges, "edges")

## OpenDRIVE (experimental)

In [ ]:
net.to_opendrive("out/monaco.xodr")
roads = ET.parse("out/monaco.xodr").findall("road")    # the roads of the OpenDRIVE file
print(f"{Path('out/monaco.xodr').stat().st_size / 1e6:.1f} MB,", len(roads), "roads,", n_edges, "edges")

## GeoPackage

In [ ]:
import duckdb

gis = net.to_gis("out/monaco.gpkg", modes=["driving"])
con = duckdb.connect()
con.load_extension("spatial")
# the number of rows of each layer, as DuckDB reads them
{layer: con.execute(f"SELECT count(*) FROM st_read('out/monaco.gpkg', layer='{layer}')").fetchone()[0]
 for layer in gis["layers"]}

## QuackOSM table and PMTiles

`quack` reads the OSM file, not the network: it gives all OSM features with their tags, as QuackOSM does. The command `duckosm quack --pmtiles` makes a base map of vector tiles in one file.

In [ ]:
quack = duckosm.quack(area.pbf, out="out", name="monaco", formats=["geoparquet"])
# the features of the table, by OSM type
from IPython.display import HTML    # to show a table with no index column

types = con.sql(f"SELECT split_part(feature_id, '/', 1) AS type, count(*) AS features FROM '{quack[0]}' GROUP BY 1 ORDER BY 1").df()
HTML(types.to_html(index=False))

In [ ]:
import subprocess

subprocess.run([sys.executable, "-m", "duckosm", "quack", str(area.pbf), "--pmtiles", "-o", "out", "--name", "monaco"],
               check=True)    # the command duckosm quack --pmtiles
header = Path("out/monaco.pmtiles").read_bytes()[:7]    # a PMTiles file starts with "PMTiles"
print(header, f"{Path('out/monaco.pmtiles').stat().st_size / 1e6:.2f} MB")